# Путь исследования: от baseline до компактных CNN и CRNN

Ноутбук показывает, как менялись решения по ходу экспериментов. По умолчанию он не обучает все модели
заново, а строит таблицы и графики по сохранённым результатам. Основные сравнения сделаны на одном
session-level split и по трём seed, если не указано другое.

In [ ]:
import copy
import random
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch import nn
from torch.utils.data import DataLoader

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dataset import AudioDataset, load_metadata, split_metadata
from src.preprocessing import CLASS_NAMES

RESULTS = ROOT / "results"
sns.set_theme(style="whitegrid")
metadata = load_metadata(ROOT / "metadata.csv")
splits = split_metadata(metadata)

## Режим выполнения и общий экспериментальный код

По умолчанию долгие этапы читают готовые CSV. Чтобы заново запустить нужный эксперимент, установите
соответствующий флаг в `True`. Ниже оставлен весь основной код: модели, аугментации, загрузка данных,
обучение с early stopping и расчёт метрик.

In [ ]:
RUN_BASELINE = False       # 6 обучений
RUN_SATURATION = False     # 42 обучения
RUN_AUGMENTATION = False   # 12 CNN + 6 CRNN обучений
RUN_SIZE_ABLATION = False  # staged search по размерам
RUN_FINAL = False          # 6 финальных обучений

SEEDS = (42, 123, 777)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
ARTIFACTS = ROOT / "artifacts" / "notebook_runs"
ARTIFACTS.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class ModelSpec:
    family: str
    channels: tuple[int, int, int]
    projection_size: int | None = None
    gru_hidden: int | None = None


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, pool):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2 if pool is True else pool))
        self.block = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.block(inputs)


class ExperimentCNN(nn.Module):
    def __init__(self, channels):
        super().__init__()
        c1, c2, c3 = channels
        self.features = nn.Sequential(
            ConvBlock(1, c1, True),
            ConvBlock(c1, c2, True),
            ConvBlock(c2, c3, False),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(c3, len(CLASS_NAMES))

    def forward(self, inputs):
        return self.classifier(self.pool(self.features(inputs)).flatten(1))


class ExperimentCRNN(nn.Module):
    def __init__(self, channels, projection_size, gru_hidden):
        super().__init__()
        c1, c2, c3 = channels
        self.cnn = nn.Sequential(
            ConvBlock(1, c1, (2, 2)),
            ConvBlock(c1, c2, (2, 2)),
            ConvBlock(c2, c3, (2, 1)),
        )
        self.projection = nn.Sequential(
            nn.Linear(c3 * 8, projection_size),
            nn.ReLU(inplace=True),
            nn.Dropout(0.10),
        )
        self.gru = nn.GRU(
            projection_size, gru_hidden, batch_first=True, bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.30), nn.Linear(2 * gru_hidden, len(CLASS_NAMES))
        )

    def forward(self, inputs):
        sequence = self.cnn(inputs).permute(0, 3, 1, 2).flatten(start_dim=2)
        _, hidden = self.gru(self.projection(sequence))
        return self.classifier(torch.cat([hidden[-2], hidden[-1]], dim=1))


def build_experiment_model(spec):
    if spec.family == "CNN":
        return ExperimentCNN(spec.channels)
    return ExperimentCRNN(spec.channels, spec.projection_size, spec.gru_hidden)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def shift_with_zeros(waveform, shift):
    if shift == 0:
        return waveform
    result = torch.zeros_like(waveform)
    if shift > 0:
        result[..., shift:] = waveform[..., :-shift]
    else:
        result[..., :shift] = waveform[..., -shift:]
    return result


class ConfiguredAugment:
    def __init__(self, config):
        self.config = copy.deepcopy(config)

    def __call__(self, waveform):
        shift = self.config["shift"]
        gain = self.config["gain"]
        noise = self.config["noise"]
        if shift["enabled"] and random.random() < shift["p"]:
            maximum = int(waveform.shape[-1] * shift["max_fraction"])
            waveform = shift_with_zeros(waveform, random.randint(-maximum, maximum))
        if gain["enabled"] and random.random() < gain["p"]:
            gain_db = random.uniform(gain["min_db"], gain["max_db"])
            waveform = waveform * (10.0 ** (gain_db / 20.0))
        if noise["enabled"] and random.random() < noise["p"]:
            signal_power = waveform.pow(2).mean().clamp_min(1e-8)
            snr_db = random.uniform(noise["min_snr_db"], noise["max_snr_db"])
            raw_noise = torch.randn_like(waveform)
            raw_power = raw_noise.pow(2).mean().clamp_min(1e-8)
            target_power = signal_power / (10.0 ** (snr_db / 10.0))
            waveform = waveform + raw_noise * torch.sqrt(target_power / raw_power)
        peak = waveform.abs().max()
        return waveform / peak if peak > 1.0 else waveform


NO_AUGMENTATION = {
    "shift": {"enabled": False, "p": 0.0, "max_fraction": 0.20},
    "gain": {"enabled": False, "p": 0.0, "min_db": -9.0, "max_db": 9.0},
    "noise": {"enabled": False, "p": 0.0, "min_snr_db": 15.0, "max_snr_db": 30.0},
}
BASELINE_AUGMENTATION = {
    "shift": {"enabled": True, "p": 0.8, "max_fraction": 0.20},
    "gain": {"enabled": True, "p": 0.3, "min_db": -6.0, "max_db": 6.0},
    "noise": {"enabled": False, "p": 0.0, "min_snr_db": 15.0, "max_snr_db": 30.0},
}
FINAL_AUGMENTATION = {
    "shift": {"enabled": True, "p": 0.5, "max_fraction": 0.20},
    "gain": {"enabled": True, "p": 0.5, "min_db": -9.0, "max_db": 9.0},
    "noise": {"enabled": False, "p": 0.0, "min_snr_db": 15.0, "max_snr_db": 30.0},
}


def make_loaders(train_frame, seed, augmentation, batch_size=BATCH_SIZE):
    train_dataset = AudioDataset(train_frame, ROOT / "data/raw", training=False)
    train_dataset.augment = ConfiguredAugment(augmentation)
    validation_dataset = AudioDataset(splits["validation"], ROOT / "data/raw", training=False)
    generator = torch.Generator().manual_seed(seed)
    common = {
        "batch_size": batch_size,
        "num_workers": 0,
        "pin_memory": torch.cuda.is_available(),
    }
    return (
        DataLoader(train_dataset, shuffle=True, generator=generator, **common),
        DataLoader(validation_dataset, shuffle=False, **common),
    )


def evaluation_loader(frame, batch_size=BATCH_SIZE):
    return DataLoader(
        AudioDataset(frame, ROOT / "data/raw", training=False),
        batch_size=batch_size, shuffle=False, num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

In [ ]:
def run_epoch(model, loader, optimizer=None, gradient_clip=False):
    training = optimizer is not None
    model.train(training)
    criterion = nn.CrossEntropyLoss()
    total_loss, targets, predictions = 0.0, [], []
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for features, labels in loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = criterion(logits, labels)
            if optimizer is not None:
                loss.backward()
                if gradient_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(labels)
            targets.extend(labels.detach().cpu().tolist())
            predictions.extend(logits.argmax(1).detach().cpu().tolist())
    labels = list(range(len(CLASS_NAMES)))
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(targets, predictions),
        "macro_f1": f1_score(targets, predictions, labels=labels, average="macro", zero_division=0),
        "targets": targets,
        "predictions": predictions,
    }


def train_one(spec_name, spec, train_frame, augmentation, seed, max_epochs, patience):
    seed_everything(seed)
    train_loader, validation_loader = make_loaders(train_frame, seed, augmentation)
    model = build_experiment_model(spec).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    safe_name = spec_name.lower().replace(" ", "_").replace("-", "")
    checkpoint = ARTIFACTS / f"{safe_name}_seed{seed}.pt"
    best_f1, best_loss, best_epoch = -np.inf, np.inf, None
    without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        run_epoch(model, train_loader, optimizer, gradient_clip=spec.family == "CRNN")
        validation = run_epoch(model, validation_loader)
        improved = (
            validation["macro_f1"] > best_f1
            or (np.isclose(validation["macro_f1"], best_f1) and validation["loss"] < best_loss)
        )
        if improved:
            best_f1, best_loss, best_epoch = validation["macro_f1"], validation["loss"], epoch
            without_improvement = 0
            torch.save(model.state_dict(), checkpoint)
        else:
            without_improvement += 1
        if without_improvement >= patience:
            break

    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    selected_validation = run_epoch(model, validation_loader)
    return model, {
        "model": spec_name,
        "family": spec.family,
        "seed": seed,
        "num_train_samples": len(train_frame),
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "best_epoch": best_epoch,
        "val_loss": selected_validation["loss"],
        "val_accuracy": selected_validation["accuracy"],
        "val_macro_f1": selected_validation["macro_f1"],
        "checkpoint": str(checkpoint),
    }


def benchmark_cpu_batch1(spec, checkpoint, warmup=100, iterations=500):
    model = build_experiment_model(spec).cpu()
    model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
    model.eval()
    sample = torch.zeros(1, 1, 64, 151)
    previous_threads = torch.get_num_threads()
    torch.set_num_threads(1)
    try:
        with torch.inference_mode():
            for _ in range(warmup):
                model(sample)
            measurements = []
            for _ in range(iterations):
                started = time.perf_counter_ns()
                model(sample)
                measurements.append((time.perf_counter_ns() - started) / 1e6)
    finally:
        torch.set_num_threads(previous_threads)
    return float(np.median(measurements))


BASELINE_SPECS = {
    "CNN Baseline": ModelSpec("CNN", (16, 32, 64)),
    "CRNN Baseline": ModelSpec("CRNN", (16, 32, 64), 128, 64),
}

print("device:", DEVICE)

## Карта экспериментов

| Этап | Вопрос | Решение для следующего этапа |
|---|---|---|
| Baseline | Даёт ли CRNN прирост относительно CNN? | Продолжить проверку обеих архитектур |
| Насыщение данных | Как меняется качество при росте train? | Полные данные для CNN; первичная проверка CRNN на 20% |
| Аугментации | Какие преобразования не портят данные и помогают модели? | Оставить shift + gain, исключить noise |
| Размер модели | Можно ли сильно уменьшить модели без потери качества? | CNN Width-8 и CRNN Micro |
| Финальное сравнение | Есть ли однозначный победитель? | Оставить обе модели для разных сценариев |

Результаты baseline на test уже просматривались во время работы. Поэтому итоговый test здесь используется
для одинакового сравнения выбранных моделей, но не является полностью нетронутым набором данных.

## 1. Baseline: CNN против CRNN

В первом эксперименте CRNN показала лучшее качество, но содержала намного больше параметров.
Поэтому дальше проверялось, можно ли уменьшить обе модели и как на результат влияет объём train.

In [ ]:
if RUN_BASELINE:
    baseline_runs = []
    test_loader = evaluation_loader(splits["test"])
    for model_name, spec in BASELINE_SPECS.items():
        for seed in SEEDS:
            model, row = train_one(
                model_name, spec, splits["train"], BASELINE_AUGMENTATION,
                seed, max_epochs=20, patience=20,
            )
            test_metrics = run_epoch(model, test_loader)
            row.update({
                "test_accuracy": test_metrics["accuracy"],
                "test_macro_f1": test_metrics["macro_f1"],
            })
            baseline_runs.append(row)

    baseline_runs = pd.DataFrame(baseline_runs)
    baseline = (
        baseline_runs.groupby(["model", "parameters"], as_index=False)
        .agg(
            test_accuracy_mean=("test_accuracy", "mean"),
            test_accuracy_std=("test_accuracy", "std"),
            test_macro_f1_mean=("test_macro_f1", "mean"),
            test_macro_f1_std=("test_macro_f1", "std"),
        )
    )
    latency_by_model = {}
    for model_name, spec in BASELINE_SPECS.items():
        selected = (
            baseline_runs[baseline_runs["model"] == model_name]
            .sort_values(["val_macro_f1", "seed"], ascending=[False, True])
            .iloc[0]
        )
        latency_by_model[model_name] = benchmark_cpu_batch1(spec, selected["checkpoint"])
    baseline["cpu_latency_b1_ms"] = baseline["model"].map(latency_by_model)
    baseline.to_csv(RESULTS / "baseline_comparison.csv", index=False)
else:
    baseline = pd.read_csv(RESULTS / "baseline_comparison.csv")
display(baseline)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=baseline, x="model", y="test_macro_f1_mean", ax=axes[0], color="#4C78A8")
axes[0].errorbar(
    range(len(baseline)), baseline["test_macro_f1_mean"],
    yerr=baseline["test_macro_f1_std"], fmt="none", color="black", capsize=4,
)
axes[0].set(title="Baseline test macro-F1", xlabel="", ylabel="Среднее ± std по seed", ylim=(0.85, 1.01))
sns.barplot(data=baseline, x="model", y="parameters", ax=axes[1], color="#F58518")
axes[1].set(title="Размер baseline-моделей", xlabel="", ylabel="Параметры")
plt.tight_layout()

**Промежуточное решение.** CRNN дала более высокий результат, но была почти в семь раз крупнее.
По одному baseline нельзя выбрать архитектуру, поэтому дальше сравнивались разные объёмы данных
и размеры моделей.

## 2. Насыщение данных

Для каждого объёма train использовалась выборка с сохранением баланса классов. Меньшие выборки
входили в более крупные, а validation-сессия не менялась.

In [ ]:
if RUN_SATURATION:
    fractions = (0.05, 0.10, 0.20, 0.40, 0.60, 0.80, 1.00)
    saturation_runs = []
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        class_orders = {}
        for class_name in CLASS_NAMES:
            indices = np.flatnonzero(splits["train"]["class"].to_numpy() == class_name)
            rng.shuffle(indices)
            class_orders[class_name] = indices

        for fraction in fractions:
            selected = []
            for class_name in CLASS_NAMES:
                indices = class_orders[class_name]
                count = len(indices) if fraction == 1.0 else max(1, round(len(indices) * fraction))
                selected.extend(indices[:count])
            subset = splits["train"].iloc[sorted(selected)].reset_index(drop=True)

            for model_name, spec in BASELINE_SPECS.items():
                model, row = train_one(
                    model_name, spec, subset, BASELINE_AUGMENTATION,
                    seed, max_epochs=20, patience=20,
                )
                row["fraction"] = fraction
                saturation_runs.append(row)

    saturation_runs = pd.DataFrame(saturation_runs)
    saturation_runs["model"] = saturation_runs["family"]
    saturation = (
        saturation_runs.groupby(["model", "fraction", "num_train_samples"], as_index=False)
        .agg(
            val_macro_f1_mean=("val_macro_f1", "mean"),
            val_macro_f1_std=("val_macro_f1", "std"),
            val_accuracy_mean=("val_accuracy", "mean"),
            val_accuracy_std=("val_accuracy", "std"),
        )
    )
    saturation.to_csv(RESULTS / "data_saturation.csv", index=False)
else:
    saturation = pd.read_csv(RESULTS / "data_saturation.csv")
display(saturation)

fig, ax = plt.subplots(figsize=(9, 5))
for model_name, frame in saturation.groupby("model"):
    frame = frame.sort_values("num_train_samples")
    ax.errorbar(
        frame["num_train_samples"], frame["val_macro_f1_mean"],
        yerr=frame["val_macro_f1_std"], marker="o", capsize=4, label=model_name,
    )
ax.set(
    title="Learning curve на фиксированной validation-сессии",
    xlabel="Количество исходных train WAV",
    ylabel="Validation macro-F1",
    ylim=(0.2, 1.03),
)
ax.legend()
plt.tight_layout()

tail = saturation[saturation["fraction"].isin([0.8, 1.0])].pivot(
    index="model", columns="fraction", values="val_macro_f1_mean"
)
tail["gain_80_to_100"] = tail[1.0] - tail[0.8]
display(tail)

**Промежуточное решение.** CRNN уже на 20% train близка к результату на полном наборе, а CNN продолжает
улучшаться с ростом данных. Поэтому варианты CRNN сначала проверялись на 20%, а лучшие затем повторно
проверялись на полном train. Варианты CNN сразу сравнивались на полном наборе. Этот результат относится
только к текущему датасету и условиям записи.

## 3. Проверка аугментаций

Сначала изменённые train-примеры прослушивались вручную. После этого варианты аугментаций сравнивались
по метрикам модели на трёх seed. Одного прослушивания оказалось недостаточно.

In [ ]:
human_validation = pd.read_csv(RESULTS / "augmentation_human_validation.csv")
human_summary = (
    human_validation.groupby(["stage", "augmentation", "human_verdict"])
    .size().rename("cases").reset_index()
)
display(human_summary)

raw_robustness = pd.read_csv(RESULTS / "augmentation_robustness_diagnostic.csv")
robustness = pd.DataFrame({
    "augmentation": ["time_shift", "gain", "noise"],
    "human_validation": "passed",
    "model_consistency": raw_robustness["prediction_consistency"],
    "macro_f1_delta": raw_robustness["delta_macro_f1"],
})
robustness["decision"] = np.where(
    (robustness["model_consistency"] >= 0.80) & (robustness["macro_f1_delta"] >= -0.05),
    "include", "exclude",
)

if RUN_AUGMENTATION:
    shift_only = copy.deepcopy(NO_AUGMENTATION)
    shift_only["shift"] = copy.deepcopy(FINAL_AUGMENTATION["shift"])
    gain_only = copy.deepcopy(NO_AUGMENTATION)
    gain_only["gain"] = copy.deepcopy(FINAL_AUGMENTATION["gain"])
    policies = {
        "NoAug": NO_AUGMENTATION,
        "TimeShiftOnly": shift_only,
        "GainOnly": gain_only,
        "ShiftAndGain": FINAL_AUGMENTATION,
    }
    augmentation_runs = []
    for policy_name, policy in policies.items():
        for seed in SEEDS:
            _, row = train_one(
                "CNN Baseline", BASELINE_SPECS["CNN Baseline"], splits["train"],
                policy, seed, max_epochs=80, patience=12,
            )
            row["policy"] = policy_name
            augmentation_runs.append(row)
    for policy_name in ("NoAug", "ShiftAndGain"):
        for seed in SEEDS:
            _, row = train_one(
                "CRNN Baseline", BASELINE_SPECS["CRNN Baseline"], splits["train"],
                policies[policy_name], seed, max_epochs=80, patience=12,
            )
            row["policy"] = policy_name
            augmentation_runs.append(row)

    augmentation_runs = pd.DataFrame(augmentation_runs)
    augmentation = (
        augmentation_runs.groupby(["family", "policy"], as_index=False)
        .agg(
            val_macro_f1_mean=("val_macro_f1", "mean"),
            val_macro_f1_std=("val_macro_f1", "std"),
            val_accuracy_mean=("val_accuracy", "mean"),
            seeds=("seed", "nunique"),
        )
        .rename(columns={"family": "architecture"})
    )
    augmentation.to_csv(RESULTS / "augmentation_comparison.csv", index=False)
    robustness.to_csv(RESULTS / "augmentation_robustness.csv", index=False)
else:
    augmentation = pd.read_csv(RESULTS / "augmentation_comparison.csv")
display(robustness)
display(augmentation)

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=augmentation, x="policy", y="val_macro_f1_mean", hue="architecture", ax=ax)
ax.set(title="Validation macro-F1 для augmentation policy", xlabel="Policy", ylabel="Среднее по seed", ylim=(0.94, 1.01))
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()

**Промежуточное решение.** Shift и gain не меняют класс записи и не ухудшают результат.
Noise прошёл ручную проверку, но совпадение предсказаний составило только `0.30`, а macro-F1 упал
на `0.764`, поэтому noise исключён. В финальном варианте только для train применяются shift до 20%
и gain ±9 dB, каждое с вероятностью 0.5. Для CRNN эта настройка проверена отдельно.

## 4. Исследование размера моделей

Ограничения в 20 эпох потенциально могло быть недостаточно для некоторых маленьких моделей. Поэтому был задан
больший максимум и добавлен early stopping: обучение завершалось, когда validation-метрика
переставала улучшаться. Пунктирные линии на графиках только соединяют проверенные варианты одной
архитектуры и не являются аппроксимацией.

In [ ]:
CNN_SPECS = {
    "Baseline": ModelSpec("CNN", (16, 32, 64)),
    "Width-12": ModelSpec("CNN", (12, 24, 48)),
    "Width-10": ModelSpec("CNN", (10, 20, 40)),
    "Width-8": ModelSpec("CNN", (8, 16, 32)),
    "Width-6": ModelSpec("CNN", (6, 12, 24)),
}
CRNN_SPECS = {
    "Baseline": ModelSpec("CRNN", (16, 32, 64), 128, 64),
    "Medium": ModelSpec("CRNN", (12, 24, 48), 96, 32),
    "Small": ModelSpec("CRNN", (8, 16, 32), 64, 32),
    "Small-7": ModelSpec("CRNN", (7, 14, 28), 56, 28),
    "Small-6": ModelSpec("CRNN", (6, 12, 24), 48, 24),
    "Small-5": ModelSpec("CRNN", (5, 10, 20), 40, 20),
    "Tiny": ModelSpec("CRNN", (4, 8, 16), 32, 16),
    "Micro": ModelSpec("CRNN", (3, 6, 12), 24, 12),
    "Nano": ModelSpec("CRNN", (2, 4, 8), 16, 8),
}

if RUN_SIZE_ABLATION:
    size_runs = []
    specs_by_key = {}

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        selected = []
        for class_name in CLASS_NAMES:
            indices = np.flatnonzero(splits["train"]["class"].to_numpy() == class_name)
            rng.shuffle(indices)
            selected.extend(indices[:max(1, round(len(indices) * 0.20))])
        crnn_subset = splits["train"].iloc[sorted(selected)].reset_index(drop=True)

        for variant, spec in CRNN_SPECS.items():
            run_name = f"CRNN {variant} screening"
            _, row = train_one(
                run_name, spec, crnn_subset, BASELINE_AUGMENTATION,
                seed, max_epochs=100, patience=15,
            )
            row.update({"stage": "screening", "train_fraction": 0.20, "variant": variant})
            size_runs.append(row)
            specs_by_key[("CRNN", "screening", variant)] = spec

        for variant in ("Baseline", "Micro", "Nano"):
            spec = CRNN_SPECS[variant]
            run_name = f"CRNN {variant} confirmation"
            _, row = train_one(
                run_name, spec, splits["train"], BASELINE_AUGMENTATION,
                seed, max_epochs=100, patience=15,
            )
            row.update({"stage": "confirmation", "train_fraction": 1.00, "variant": variant})
            size_runs.append(row)
            specs_by_key[("CRNN", "confirmation", variant)] = spec

        for variant, spec in CNN_SPECS.items():
            run_name = f"CNN {variant} confirmation"
            _, row = train_one(
                run_name, spec, splits["train"], BASELINE_AUGMENTATION,
                seed, max_epochs=100, patience=15,
            )
            row.update({"stage": "confirmation", "train_fraction": 1.00, "variant": variant})
            size_runs.append(row)
            specs_by_key[("CNN", "confirmation", variant)] = spec

    size_runs = pd.DataFrame(size_runs)
    size_ablation = (
        size_runs.groupby(["family", "stage", "train_fraction", "variant", "parameters"], as_index=False)
        .agg(
            val_macro_f1_mean=("val_macro_f1", "mean"),
            val_macro_f1_std=("val_macro_f1", "std"),
        )
    )
    latency_values = []
    for row in size_ablation.itertuples(index=False):
        candidates = size_runs[
            (size_runs["family"] == row.family)
            & (size_runs["stage"] == row.stage)
            & (size_runs["variant"] == row.variant)
        ].sort_values(["val_macro_f1", "seed"], ascending=[False, True])
        checkpoint = candidates.iloc[0]["checkpoint"]
        spec = specs_by_key[(row.family, row.stage, row.variant)]
        latency_values.append(benchmark_cpu_batch1(spec, checkpoint))
    size_ablation["cpu_latency_b1_ms"] = latency_values
    size_ablation["gpu_latency_b1_ms"] = np.nan
    size_ablation.to_csv(RESULTS / "model_size_ablation.csv", index=False)
else:
    size_ablation = pd.read_csv(RESULTS / "model_size_ablation.csv")
confirmation = size_ablation[size_ablation["stage"] == "confirmation"].copy()
display(confirmation.sort_values(["family", "parameters"], ascending=[True, False]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
family_colors = dict(zip(
    sorted(confirmation["family"].unique()),
    sns.color_palette(n_colors=confirmation["family"].nunique()),
))
for family, frame in confirmation.groupby("family"):
    by_parameters = frame.sort_values("parameters")
    axes[0].errorbar(
        frame["parameters"], frame["val_macro_f1_mean"],
        yerr=frame["val_macro_f1_std"], marker="o", linestyle="none", capsize=4,
        color=family_colors[family], label=family,
    )
    axes[0].plot(
        by_parameters["parameters"], by_parameters["val_macro_f1_mean"],
        linestyle="--", linewidth=1.2, alpha=0.65, color=family_colors[family],
    )
    for row in frame.itertuples():
        axes[0].annotate(row.variant, (row.parameters, row.val_macro_f1_mean), xytext=(4, 4), textcoords="offset points")
axes[0].set_xscale("log")
axes[0].set(title="Качество и число параметров", xlabel="Параметры, log scale", ylabel="Validation macro-F1")
axes[0].legend()

for family, frame in confirmation.groupby("family"):
    by_latency = frame.sort_values("cpu_latency_b1_ms")
    axes[1].plot(
        by_latency["cpu_latency_b1_ms"], by_latency["val_macro_f1_mean"],
        linestyle="--", linewidth=1.2, alpha=0.65, color=family_colors[family],
    )
sns.scatterplot(
    data=confirmation, x="cpu_latency_b1_ms", y="val_macro_f1_mean",
    hue="family", style="variant", palette=family_colors, s=100, ax=axes[1],
)
axes[1].set(title="Качество и CPU latency, batch=1", xlabel="Медиана, мс", ylabel="Validation macro-F1")
plt.tight_layout()

**Промежуточное решение.** CNN Width-8 в 3.88 раза меньше baseline; средний validation F1 снизился
с `1.0000` до `0.9833`, что принято как компромисс ради уменьшения модели. Width-6 работает ещё менее
стабильно. CRNN Micro в 27 раз меньше baseline и сохраняет качество на полном train, а у Nano качество
начинает снижаться. Эти опыты проводились до окончательного выбора
аугментаций, поэтому они подтверждают выбор компактных моделей, но не гарантируют, что найдены лучшие
возможные настройки.

## 5. Финальное сравнение компактных моделей

CNN Width-8 и CRNN Micro обучены в одинаковых финальных условиях. Для каждой модели сделано три запуска.
Checkpoint выбирался по validation, без учёта test. Эксперименты проводились на компьютере с
AMD Ryzen 5 2600 и NVIDIA GeForce RTX 2060 Super. В итоговом сравнении приведена CPU latency
при `batch=1`; на другом процессоре значения будут отличаться.

In [ ]:
FINAL_SPECS = {
    "CNN Width-8": ModelSpec("CNN", (8, 16, 32)),
    "CRNN Micro": ModelSpec("CRNN", (3, 6, 12), 24, 12),
}

if RUN_FINAL:
    metric_rows, prediction_frames, report_rows = [], [], []
    test_loader = evaluation_loader(splits["test"])
    selected_checkpoints = {}

    for model_name, spec in FINAL_SPECS.items():
        model_rows = []
        for seed in SEEDS:
            model, row = train_one(
                model_name, spec, splits["train"], FINAL_AUGMENTATION,
                seed, max_epochs=100, patience=15,
            )
            test_metrics = run_epoch(model, test_loader)
            labels = list(range(len(CLASS_NAMES)))
            row.update({
                "test_loss": test_metrics["loss"],
                "test_accuracy": test_metrics["accuracy"],
                "test_macro_precision": precision_score(
                    test_metrics["targets"], test_metrics["predictions"],
                    labels=labels, average="macro", zero_division=0,
                ),
                "test_macro_recall": recall_score(
                    test_metrics["targets"], test_metrics["predictions"],
                    labels=labels, average="macro", zero_division=0,
                ),
                "test_macro_f1": test_metrics["macro_f1"],
            })
            metric_rows.append(row)
            model_rows.append(row)

            report = classification_report(
                test_metrics["targets"], test_metrics["predictions"],
                labels=labels, target_names=CLASS_NAMES,
                output_dict=True, zero_division=0,
            )
            for class_name in CLASS_NAMES:
                report_rows.append({
                    "model": model_name,
                    "seed": seed,
                    "class": class_name,
                    "precision": report[class_name]["precision"],
                    "recall": report[class_name]["recall"],
                    "f1": report[class_name]["f1-score"],
                    "support": report[class_name]["support"],
                })

            prediction_frame = splits["test"][["path", "session", "class"]].copy()
            prediction_frame = prediction_frame.rename(columns={"path": "file", "class": "true_class"})
            prediction_frame.insert(0, "seed", seed)
            prediction_frame.insert(0, "model", model_name)
            prediction_frame["y_true"] = test_metrics["targets"]
            prediction_frame["y_pred"] = test_metrics["predictions"]
            prediction_frame["predicted_class"] = [CLASS_NAMES[index] for index in test_metrics["predictions"]]
            prediction_frame["correct"] = prediction_frame["y_true"] == prediction_frame["y_pred"]
            prediction_frames.append(prediction_frame)

        representative = (
            pd.DataFrame(model_rows)
            .sort_values(["val_macro_f1", "val_loss", "seed"], ascending=[False, True, True])
            .iloc[0]
        )
        selected_checkpoints[model_name] = representative["checkpoint"]

    metrics_by_seed = pd.DataFrame(metric_rows)
    aggregation = {
        "parameters": ("parameters", "first"),
        "val_macro_f1_mean": ("val_macro_f1", "mean"),
        "val_macro_f1_std": ("val_macro_f1", "std"),
        "test_loss_mean": ("test_loss", "mean"),
        "test_loss_std": ("test_loss", "std"),
        "test_accuracy_mean": ("test_accuracy", "mean"),
        "test_accuracy_std": ("test_accuracy", "std"),
        "test_macro_precision_mean": ("test_macro_precision", "mean"),
        "test_macro_precision_std": ("test_macro_precision", "std"),
        "test_macro_recall_mean": ("test_macro_recall", "mean"),
        "test_macro_recall_std": ("test_macro_recall", "std"),
        "test_macro_f1_mean": ("test_macro_f1", "mean"),
        "test_macro_f1_std": ("test_macro_f1", "std"),
        "best_epoch_mean": ("best_epoch", "mean"),
        "best_epoch_std": ("best_epoch", "std"),
    }
    metrics = metrics_by_seed.groupby("model", as_index=False).agg(**aggregation)
    metrics_by_seed_export = metrics_by_seed.copy()
    metrics_by_seed_export["checkpoint"] = metrics_by_seed_export["checkpoint"].map(
        lambda value: Path(value).relative_to(ROOT).as_posix()
    )
    metrics_by_seed_export.to_csv(RESULTS / "metrics_by_seed.csv", index=False)
    metrics.to_csv(RESULTS / "metrics_summary.csv", index=False)

    per_class_runs = pd.DataFrame(report_rows)
    per_class = (
        per_class_runs.groupby(["model", "class"], as_index=False)
        .agg(
            precision_mean=("precision", "mean"), precision_std=("precision", "std"),
            recall_mean=("recall", "mean"), recall_std=("recall", "std"),
            f1_mean=("f1", "mean"), f1_std=("f1", "std"), support=("support", "first"),
        )
    )
    per_class.to_csv(RESULTS / "per_class_metrics_summary.csv", index=False)

    all_predictions = pd.concat(prediction_frames, ignore_index=True)
    for model_name, filename in {
        "CNN Width-8": "predictions_cnn.csv",
        "CRNN Micro": "predictions_crnn.csv",
    }.items():
        all_predictions[all_predictions["model"] == model_name].to_csv(RESULTS / filename, index=False)
    all_predictions[~all_predictions["correct"]].to_csv(RESULTS / "errors.csv", index=False)

    complexity_rows, latency_rows = [], []
    for model_name, spec in FINAL_SPECS.items():
        checkpoint = Path(selected_checkpoints[model_name])
        model = build_experiment_model(spec)
        parameters = sum(parameter.numel() for parameter in model.parameters())
        complexity_rows.append({
            "model": model_name,
            "parameters": parameters,
            "float32_parameter_size_bytes": parameters * 4,
            "state_dict_size_bytes": checkpoint.stat().st_size,
            "state_dict_size_kb": checkpoint.stat().st_size / 1024,
            "state_dict_size_mb": checkpoint.stat().st_size / 1024 ** 2,
        })
        representative_seed = int(
            metrics_by_seed[
                (metrics_by_seed["model"] == model_name)
                & (metrics_by_seed["checkpoint"] == str(checkpoint))
            ].iloc[0]["seed"]
        )
        latency_rows.append({
            "model": model_name, "representative_seed": representative_seed,
            "device": "cpu", "batch_size": 1,
            "throughput_samples_per_sec": np.nan,
            "latency_median_ms": benchmark_cpu_batch1(spec, checkpoint),
            "latency_p95_ms": np.nan, "latency_mean_ms": np.nan,
            "latency_std_ms": np.nan, "measurements": 500,
        })
    complexity = pd.DataFrame(complexity_rows)
    latency = pd.DataFrame(latency_rows)
    complexity.to_csv(RESULTS / "complexity.csv", index=False)
    latency.to_csv(RESULTS / "latency.csv", index=False)
else:
    metrics = pd.read_csv(RESULTS / "metrics_summary.csv")
    complexity = pd.read_csv(RESULTS / "complexity.csv")
    latency = pd.read_csv(RESULTS / "latency.csv")
    per_class = pd.read_csv(RESULTS / "per_class_metrics_summary.csv")

In [ ]:
cpu_b1 = latency.query("device == 'cpu' and batch_size == 1")[["model", "latency_median_ms"]]
final_table = (
    metrics.merge(complexity, on=["model", "parameters"])
    .merge(cpu_b1, on="model")
    [[
        "model", "parameters", "state_dict_size_kb",
        "test_accuracy_mean", "test_accuracy_std",
        "test_macro_f1_mean", "test_macro_f1_std",
        "latency_median_ms",
    ]]
)
display(final_table)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].bar(metrics["model"], metrics["test_macro_f1_mean"], yerr=metrics["test_macro_f1_std"], capsize=5)
axes[0].set(title="Test macro-F1", ylabel="Среднее ± std по seed", ylim=(0.88, 1.02))
axes[1].bar(complexity["model"], complexity["parameters"])
axes[1].set(title="Параметры", ylabel="Количество")
axes[2].bar(cpu_b1["model"], cpu_b1["latency_median_ms"])
axes[2].set(title="CPU latency, batch=1", ylabel="Медиана, мс")
for axis in axes:
    axis.tick_params(axis="x", rotation=15)
plt.tight_layout()

## 6. Ошибки и качество по классам

Результаты по трём seed показывают не только общее число ошибок, но и то, какие классы модели путают
чаще всего. Каждый test-файл учитывается один раз для каждого seed.

In [ ]:
predictions = pd.concat([
    pd.read_csv(RESULTS / "predictions_cnn.csv"),
    pd.read_csv(RESULTS / "predictions_crnn.csv"),
], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, (model_name, frame) in zip(axes, predictions.groupby("model", sort=False)):
    matrix = confusion_matrix(frame["y_true"], frame["y_pred"], labels=range(len(CLASS_NAMES)))
    ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES).plot(ax=axis, values_format="d", colorbar=False)
    axis.set_title(f"{model_name}: сумма по трём seed")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=per_class, x="class", y="f1_mean", hue="model", ax=ax)
ax.set(title="Test F1 по классам", xlabel="Класс", ylabel="Среднее по seed", ylim=(0.75, 1.02))
plt.tight_layout()

error_counts = predictions.assign(error=~predictions["correct"].astype(bool)).groupby("model")["error"].sum()
display(error_counts.rename("errors across three seeds").to_frame())

### Краткий анализ ошибок

В сумме по трём seed CNN допустила 13 ошибочных предсказаний. Чаще всего модель путала
`knock → clap` — 4 случая. Также встречались ошибки `whistle → sing` и `clap → knock`
— по 3 случая, `knock → whistle` — 2 случая и `knock → sing` — 1 случай.

CRNN допустила две ошибки: `knock → sing` и `knock → clap`. Они относятся к seed 123 и 777;
для seed 42 ошибок не было. Это показывает, что результат зависит от конкретного запуска обучения.

В текущем эксперименте для CNN самым сложным оказался класс `knock`. CRNN ошибалась реже,
но двух ошибок недостаточно, чтобы делать общий вывод о характере её ошибок. Кроме того, проверка
проведена на одной test-сессии, поэтому найденные закономерности могут измениться на новых записях.

## Итоговый вывод

В ходе экспериментов большие baseline-модели удалось сократить до двух вариантов примерно по 6 тыс.
параметров. CNN Width-8 и CRNN Micro выбраны после проверки объёма данных, аугментаций и размера моделей.

По трём seed CRNN показала более высокое среднее test-качество: macro-F1 `0.9934 ± 0.0058` против
`0.9576 ± 0.0215` у CNN и 2 ошибки против 13. В этом запуске CRNN была лучше на каждом seed:
разность macro-F1 составила `+0.0177`, `+0.0471` и `+0.0425` для seed 42, 123 и 777 соответственно.
При этом трёх запусков и одной test-сессии всё ещё недостаточно для общего вывода о превосходстве CRNN.

При почти одинаковом числе параметров CNN примерно в `2.61×` быстрее на CPU при `batch=1`
(`1.43` против `3.73` мс). Одной лучшей модели нет: CRNN подходит,
если важнее качество, а CNN — если важнее скорость и простота. Поэтому в проекте сохранены обе модели.

Для сохранения весов checkpoint выбирался только по validation: seed 777 для CNN и seed 123 для CRNN.
Их test macro-F1 (`0.9473` и `0.9903`) ниже лучших test-результатов этих моделей. Это нормально и показывает,
что test не использовался для выбора checkpoint.

Главное ограничение — одна независимая test-сессия из 110 файлов. На этом этапе эксперименты остановлены
специально: их достаточно, чтобы обосновать компактные модели и показать разницу между ними, но недостаточно
для окончательного вывода о лучшей архитектуре. Дальше можно увеличить число seed, добавить новые сессии,
провести group cross-validation и подробнее разобрать ошибки между классами.